### SETUP
Utilizaremos el conjunto de datos de entrenamiento ISIC 2019 y sus etiquetas, y que genere dos subconjuntos de datos a
partir de dicho conjunto: el primero (será el conjunto de entrenamiento de nuestro modelo) contendrá 100 imágenes de cada clase, mientras que el segundo (será el conjunto de test de nuestro modelo) contendrá 10 imágenes de cada clase.

In [ ]:
# Importaciones
import os
import shutil
import pandas as pd
import zipfile
import requests
from sklearn.model_selection import train_test_split

### Configuración

In [ ]:
#Configuración de rutas y URLs según el enunciado de la práctica
URL_DATA = "https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_Input.zip"
URL_LABELS = "https://isic-challenge-data.s3.amazonaws.com/2019/ISIC_2019_Training_GroundTruth.csv"

# Nombre de carpetas
BASE_DIR = "dataset_isic"
RAW_DIR = "raw_data"  # Donde descargaremos el zip original
TRAIN_DIR = os.path.join(BASE_DIR, "train")
TEST_DIR = os.path.join(BASE_DIR, "test")

# Crear directorios iniciales
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(TRAIN_DIR, exist_ok=True)
os.makedirs(TEST_DIR, exist_ok=True)

print("Directorios configurados.")

## Descarga y extracción de datos

In [ ]:
def descargar_archivo(url, destino):
    if os.path.exists(destino):
        print(f"El archivo {destino} ya existe. Saltando descarga.")
        return
    print(f"Descargando {url}...")
    response = requests.get(url, stream=True)
    with open(destino, 'wb') as f:
        shutil.copyfileobj(response.raw, f)
    print("Descarga completada.")

def descomprimir_zip(ruta_zip, destino_extract):
    print("Descomprimiento imágenes... (esto puede tardar un poco)")
    with zipfile.ZipFile(ruta_zip, 'r') as zip_ref:
        zip_ref.extractall(destino_extract)
    print("Descompresión finalizada.")

# Ejecutar descarga (CSV y ZIP)
csv_path = os.path.join(RAW_DIR, "ground_truth.csv")
zip_path = os.path.join(RAW_DIR, "images.zip")

descargar_archivo(URL_LABELS, csv_path)
descargar_archivo(URL_DATA, zip_path)

# La carpeta donde se descomprimen las imágenes originales
images_extract_path = os.path.join(RAW_DIR, "ISIC_2019_Training_Input")
if not os.path.exists(images_extract_path):
    descomprimir_zip(zip_path, RAW_DIR)
else:
    print("Imágenes ya descomprimidas anteriormente.")

### Procesamiento de datos y creación de conjuntos de entrenamiento y test

In [ ]:
# Cargar el CSV
df = pd.read_csv(csv_path)

# Las clases son todas las columnas menos la primera ('image') y la última ('UNK') si existe
clases = df.columns[1:-1] if 'UNK' in df.columns else df.columns[1:]
print(f"Clases detectadas: {list(clases)}")

# TODO: Lógica principal de movimiento de archivos
for clase in clases:
    print(f"\nProcesando clase: {clase}")

    # Crear carpetas específicas para YOLO (dataset/train/MEL, dataset/test/MEL, etc.)
    os.makedirs(os.path.join(TRAIN_DIR, clase), exist_ok=True)
    os.makedirs(os.path.join(TEST_DIR, clase), exist_ok=True)

    # Filtrar imágenes que pertenecen a esta clase (valor == 1.0)
    imagenes_clase = df[df[clase] == 1.0]['image'].tolist()

    # [cite_start]Seleccionar aleatoriamente 110 imágenes (100 train + 10 test) [cite: 27]
    if len(imagenes_clase) < 110:
        print(f"⚠️ ADVERTENCIA: La clase {clase} tiene menos de 110 imágenes ({len(imagenes_clase)}). Se usarán todas.")
        seleccion = imagenes_clase
    else:
        # Mezclamos y cogemos 110
        seleccion = pd.Series(imagenes_clase).sample(n=110, random_state=42).tolist()

    # Dividir: Las primeras 100 para train, las siguientes 10 para test
    imgs_train = seleccion[:100]
    imgs_test = seleccion[100:110]

    # Función auxiliar para mover/copiar
    def mover_imagenes(lista_imgs, carpeta_destino):
        count = 0
        for img_name in lista_imgs:
            src = os.path.join(images_extract_path, img_name + ".jpg")
            dst = os.path.join(carpeta_destino, img_name + ".jpg")

            if os.path.exists(src):
                shutil.copy(src, dst)
                count += 1
        return count

    n_train = mover_imagenes(imgs_train, os.path.join(TRAIN_DIR, clase))
    n_test = mover_imagenes(imgs_test, os.path.join(TEST_DIR, clase))

    print(f" -> {n_train} imágenes movidas a TRAIN")
    print(f" -> {n_test} imágenes movidas a TEST")

print("\n Setup finalizado. Estructura de datos lista para YOLO.")